1. CausalAttention 클래스에서 역으로 멀트해드 어텐션을 구현해본다.
    - 키벡터를 멀티해드로 합성곱 하기 위해 작업한다.

        ![문맥 차원](image/03-06-example1.png)

    - 쿼리벡터를 멀티헤드로 합성곱 하기 위해 작업한다.

        ![문맥 차원](image/03-06-example2.png)
  

In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
batch = torch.stack((inputs, inputs), dim=0)

In [117]:
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        
         # 가중치 고정을 위해 직접 텐서로 초기화
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
            
        self.dropout = torch.nn.Dropout(dropout)  # 드롭아웃 확률
        
        # 미래의 단어를 마스킹하는 상삼각 행렬 생성
        # register_buffer 메서드에 지정한 텐서를 모델과 함께 적절한 장치로 자동 이동시킨다.
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        
    def forward(self, x):
        # x의 크기: (batch, num_tokens, d_in)     
        b, num_tokens, d_in = x.shape
        # 입력 x에 대해 쿼리, 키, 값 벡터 계산
        queries = self.W_query(x) # 모든 입력 토큰에 대한 쿼리 벡터 계산
        keys = self.W_key(x)     # 모든 입력 토큰에 대한 키 벡터 계산
        values = self.W_value(x)  # 모든 입력 토큰에 대한 값 벡터 계산
        #print("keys1:", keys.shape)
        #print("queries : ",queries) # (2, 6, 2)
        #print("queries.shape : ",queries.shape) # (2, 6, 2)
        #print("self.x.shape : ",x.shape) # (2, 6, 2)
        #print("self.W_value.shape : ",self.W_value.weight.shape) # (2, 6, 2)
        #print("values.shape : ",values.shape) # (2, 6, 2)
        
        # 첫 번재 차원인 배치 차원은 그대로 유지하면서 두 번재 차원과 세 번째 차원을 바꾼다.
        attn_scores = queries @ keys.transpose(1, 2) 
        
        # 마스크를 사용하여 어텐션 점수에서 주 대각선 위의 값을 -inf로 만듦
        #   - masked_fill: mask 행렬의 1인 위치에 -inf를 채움
        #   - "_"로 끝나는 메서드는 불필요한 메모리 복사를 피하기 위해 인플레이스 연산을 수행한다.
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], float('-inf'))

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # 어텐션 가중치 계산

        # 드롭아웃 적용
        attn_weights = self.dropout(attn_weights)
        
        #print("values.shape : ",values.shape) # (2, 6, 2)
        context_vecs = attn_weights @ values # 문맥 벡터 계산
        #print("attn_weights.shape : ",attn_weights.shape) # (2, 6, 6)
        
        
        # context_vecs가 (2, 6, 2) --> (2, 6, 4) 가되어야한다.
        # attn_weights (2, 6, 6) @ values (2, 6, 2) --> (2, 6, 2)
        #   --> attn_weights (2, 6, 6) @ values (2, 6, 4) --> (2, 6, 4)
        
        # values (2, 6, 2) --> (2, 6, 4)
        #   - W_value (3, 2) --> (3, 4)
        #Multi_W_value = nn.Linear(3, 4, bias=False)
        #print("Multi_W_value.weight.shape : ",Multi_W_value.weight.shape) # (4, 3)
        
        #print("x.shape : ",x.shape) # (2, 6, 3)
        #Multi_values = Multi_W_value(x)
        #print("Multi_values.shape : ",Multi_values.shape) # (2, 6, 4)
        
        #multi_context_vecs = attn_weights @ Multi_values # 문맥 벡터 계산
        #print("multi_context_vecs : ",multi_context_vecs) # (2, 6, 4)
        #print("multi_context_vecs.shape : ",multi_context_vecs.shape) # (2, 6, 4)
        
        multi_keys = self.W_key(x)
        #print("x.shape : ",x.shape) # (2, 6, 3)
        #print("self.W_key.weight.shape : ",self.W_key.weight.shape) # (2, 3)
        #print("multi_keys.shape : ",multi_keys.shape) # (2, 6, 4) 
        
        b, num_tokens, d_in = x.shape
        head_dim = self.d_out // 2
        multi_keys_v = multi_keys.view(b, num_tokens, 2, head_dim)
        #print("keys2:", multi_keys_v.shape)
        #print("multi_keys_v.shape : ",multi_keys_v.shape) # (1, 6, 2, 2)
        #print("multi_keys_v : ",multi_keys_v) # (1, 6, 2, 2)
        
        multi_keys_t = multi_keys_v.transpose(1, 2)
        #print("keys3:", multi_keys_t.shape)
        #print("multi_keys_t.shape : ",multi_keys_t.shape) # (1, 2, 6, 2)
        #print("multi_keys_t : ",multi_keys_t) # (1, 2,
        
        multi_keys_t2 = multi_keys_t.transpose(2, 3)
        #print("keys4:", multi_keys_t2.shape)
        #print("multi_keys_t2.shape : ",multi_keys_t2.shape) # (1, 6, 2, 2)
        #print("multi_keys_t2 : ",multi_keys_t2) # (1, 6, 2, 2)
        
        #print("queries.shape : ",queries.shape) # (2, 6, 2)
        #print("queries : ",queries) # (2, 6, 2)
        queries = queries.view(b, num_tokens, 2, head_dim)

        #print("queries.shape : ",queries.shape) # (2, 6, 2)
        #print("queries : ",queries) # (2, 6, 2)
        queries = queries.transpose(1, 2)

        print("queries.shape : ",queries.shape) # (2, 6, 2)
        print("queries : ",queries) # (2, 6, 2)
        multi_attn_scores = queries @ multi_keys_t2 
        #print("multi_attn_scores.shape : ",multi_attn_scores.shape) # (2, 6, 6)
        #print("multi_attn_scores : ",multi_attn_scores) # (2, 6, 6)
        return context_vecs

In [118]:
torch.manual_seed(123)
context_length = batch.shape[1] # 컨텍스트 길이
#print("컨텍스트 길이: ", context_length)

# 입력 임베딩 크기 3, 출력 임베딩 크기 2
ca = CausalAttention(d_in=3, d_out=2, context_length=context_length, dropout=0.0) 

context_ves = ca(batch) # 첫 번째 문장에 대한 문맥 벡터 계산
#print("문맥 벡터:\n", context_ves)
#print("문맥 벡터 크기: ", context_ves.shape)

queries.shape :  torch.Size([2, 2, 6, 1])
queries :  tensor([[[[-0.3536],
          [-0.3021],
          [-0.3015],
          [-0.1353],
          [-0.2052],
          [-0.1542]],

         [[ 0.3965],
          [-0.0289],
          [-0.0232],
          [-0.0978],
          [ 0.0870],
          [-0.1499]]],


        [[[-0.3536],
          [-0.3021],
          [-0.3015],
          [-0.1353],
          [-0.2052],
          [-0.1542]],

         [[ 0.3965],
          [-0.0289],
          [-0.0232],
          [-0.0978],
          [ 0.0870],
          [-0.1499]]]], grad_fn=<TransposeBackward0>)
